# Project FORESIGHT — 01 Exploratory Data Analysis

This notebook analyzes the processed sales and inventory data for **NorthBay Living** to identify demand seasonality, trends, top movers, and dead-stock SKUs.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Load Cleaned Dataset

In [ ]:
processed_path = "../data/processed/clean_sales.csv"
if not os.path.exists(processed_path):
    # Fallback to local path if running from root
    processed_path = "data/processed/clean_sales.csv"
    
df = pd.read_csv(processed_path)
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Weekend Seasonality Analysis
Verify if weekend demand is significantly higher than weekday demand.

In [ ]:
# Average sales by day of week (0=Monday, 6=Sunday)
dow_sales = df.groupby("day_of_week")["units_sold"].mean().reset_index()
dow_map = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}
dow_sales["day_name"] = dow_sales["day_of_week"].map(dow_map)

plt.figure(figsize=(8, 4))
sns.barplot(data=dow_sales, x="day_name", y="units_sold", palette="viridis")
plt.title("Average Units Sold by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Average Units")
plt.show()

weekend_avg = df[df["day_of_week"].isin([4, 5, 6])]["units_sold"].mean()
weekday_avg = df[~df["day_of_week"].isin([4, 5, 6])]["units_sold"].mean()
print(f"Average Weekend Sales: {weekend_avg:.2f}")
print(f"Average Weekday Sales: {weekday_avg:.2f}")
print(f"Weekend Lift: {((weekend_avg / weekday_avg) - 1)*100:.1f}%")

## 3. Category Seasonality Analysis
Analyze demand patterns across seasons and product categories.

In [ ]:
cat_season = df.groupby(["category", "season"])["units_sold"].mean().reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=cat_season, x="category", y="units_sold", hue="season", palette="muted")
plt.title("Average Sales by Category and Season")
plt.ylabel("Average Units")
plt.xticks(rotation=15)
plt.legend(title="Season")
plt.show()

## 4. SKU Classification (Top Movers vs. Slow Moving / Dead Stock)
Identify SKUs accounting for most sales and SKUs that are slow-moving.

In [ ]:
# SKU level revenue and demand CV
sku_summary = df.groupby("sku_id").agg(
    total_units=("units_sold", "sum"),
    total_revenue=("revenue", "sum"),
    mean_units=("units_sold", "mean"),
    std_units=("units_sold", "std")
).reset_index()

sku_summary["cv"] = sku_summary["std_units"] / sku_summary["mean_units"]
sku_summary = sku_summary.sort_values(by="total_revenue", ascending=False).reset_index(drop=True)

print("--- TOP 10 MOVERS BY REVENUE ---")
print(sku_summary.head(10))

print("\n--- SLOW-MOVING / DEAD STOCK (BOTTOM 10 BY REVENUE) ---")
print(sku_summary.tail(10))

# Pareto principle check: does top 20% SKUs generate 80% revenue?
sku_summary["cum_rev_pct"] = sku_summary["total_revenue"].cumsum() / sku_summary["total_revenue"].sum()
sku_summary["cum_sku_pct"] = (sku_summary.index + 1) / len(sku_summary)

plt.figure(figsize=(8, 4))
plt.plot(sku_summary["cum_sku_pct"], sku_summary["cum_rev_pct"], marker="o", color="b")
plt.axhline(y=0.8, color="r", linestyle="--", label="80% Revenue")
plt.axvline(x=0.2, color="g", linestyle="--", label="20% SKUs")
plt.title("Pareto Chart of Cumulative Revenue vs SKUs")
plt.xlabel("Share of SKUs")
plt.ylabel("Share of Revenue")
plt.legend()
plt.show()

top_20_rev_share = sku_summary[sku_summary["cum_sku_pct"] <= 0.20]["total_revenue"].sum() / sku_summary["total_revenue"].sum()
print(f"Revenue share of top 20% SKUs: {top_20_rev_share*100:.1f}%")

## 5. Promotion Effectiveness
Evaluate sales uplift during promotional periods.

In [ ]:
promo_sales = df.groupby("promo_flag")["units_sold"].mean().reset_index()
print(promo_sales)

plt.figure(figsize=(6, 4))
sns.barplot(data=promo_sales, x="promo_flag", y="units_sold", palette="Set2")
plt.title("Average Daily Sales: Promo vs Non-Promo")
plt.ylabel("Units Sold")
plt.xticks([0, 1], ["Regular Day", "Promo Day"])
plt.show()

uplift = ((promo_sales.loc[1, "units_sold"] / promo_sales.loc[0, "units_sold"]) - 1) * 100
print(f"Promotional Lift: {uplift:.1f}%")